# Libraries

In [1]:
# core libraries
import numpy as np
import pandas as pd

# preprocessing

pd.set_option('display.max_columns',None)

# Data

## Data Dictionary

In [2]:
pd.read_csv('/content/drive/MyDrive/Data Science Projects/Fraud Model Development/data/data_dictionary.csv').head(50)

,column,description,type
0,transaction_id,Unique transaction identifier,integer
1,account_id,Account associated with transaction,integer
2,customer_id,Customer associated with transaction,integer
3,timestamp,Transaction timestamp,datetime
4,amount,Transaction amount,float
5,merchant_id,Merchant identifier,integer
6,merchant_category,Merchant category,categorical
7,channel,Transaction channel,categorical
8,authentication_method,Authentication method used,categorical
9,device_id,Device identifier,integer


## Transactions (Base Table)

In [3]:
transaction_df = pd.read_csv('/content/drive/MyDrive/Data Science Projects/Fraud Model Development/data/transactions.csv')
print(transaction_df.shape)
transaction_df.head()

(2000000, 17)


,transaction_id,account_id,customer_id,timestamp,amount,merchant_id,merchant_category,channel,authentication_method,device_id,ip_id,transaction_state,fraud_label,fraud_type,fraud_discovery_date,existing_model_score,existing_model_prediction
0,1828402,36178,95737,2024-01-08 07:01:49,61.74,1345,restaurant,ECOMMERCE,contactless,55024.0,104754.0,TN,0,legitimate,NaN,0.225364,0
1,1200072,52330,32499,2025-02-13 10:55:55,19.58,6159,grocery,ECOMMERCE,3ds,41220.0,83926.0,NY,0,legitimate,NaN,0.047296,0
2,194850,16654,59925,2024-10-02 14:59:28,337.28,8986,grocery,MOBILE,password,6709.0,17431.0,LA,0,legitimate,NaN,0.043783,0
3,1629055,100442,74540,2024-09-09 06:32:08,43.37,9382,online_marketplace,POS,contactless,4769.0,106315.0,MD,0,legitimate,NaN,0.014600,0
4,191145,88323,83760,2024-05-26 05:09:17,17.83,6994,restaurant,PHONE,contactless,23.0,1533.0,OK,0,legitimate,NaN,0.169942,0


## Accounts

In [4]:
account_df = pd.read_csv('/content/drive/MyDrive/Data Science Projects/Fraud Model Development/data/accounts.csv')
print(account_df.shape)
account_df.head()

(130000, 5)


,account_id,customer_id,account_age_days,account_type,credit_limit
0,1,66128,357,debit_card,NaN
1,2,79474,1454,credit_card,27731.42
2,3,2440,1295,savings,NaN
3,4,28259,1205,checking,NaN
4,5,44530,1255,credit_card,3849.53


## Customers

In [5]:
customer_df = pd.read_csv('/content/drive/MyDrive/Data Science Projects/Fraud Model Development/data/customers.csv')
print(customer_df.shape)
customer_df.head()

(100000, 5)


,customer_id,age,annual_income,home_state,customer_tenure_days
0,1,46,26950.41,NE,1079
1,2,27,354802.25,RI,1059
2,3,53,241070.32,PA,182
3,4,55,43765.76,GA,410
4,5,18,212383.14,MS,1160


## Devices

In [6]:
device_df = pd.read_csv('/content/drive/MyDrive/Data Science Projects/Fraud Model Development/data/devices.csv')
print(device_df.shape)
device_df.head()

(90000, 3)


,device_id,device_type,device_age_days
0,1,android,1181
1,2,mac,346
2,3,iphone,580
3,4,android,1496
4,5,android,533


## IPs

In [7]:
ip_df = pd.read_csv('/content/drive/MyDrive/Data Science Projects/Fraud Model Development/data/ips.csv')
print(ip_df.shape)
ip_df.head()

(140000, 2)


,ip_id,ip_type
0,1,mobile
1,2,mobile
2,3,residential
3,4,mobile
4,5,residential


## Merchants

In [8]:
merchant_df = pd.read_csv('/content/drive/MyDrive/Data Science Projects/Fraud Model Development/data/merchants.csv')
print(merchant_df.shape)
merchant_df.head()

(10000, 2)


,merchant_id,merchant_category
0,1,subscription
1,2,financial_services
2,3,subscription
3,4,restaurant
4,5,jewelry


# Join DataFrames

In [9]:
df = transaction_df.merge(account_df[[col for col in account_df.columns if col != 'customer_id']],on='account_id',how='left')
df = df.merge(customer_df,on='customer_id',how='left')
df = df.merge(device_df,on='device_id',how='left')
df = df.merge(ip_df,on='ip_id',how='left')
df = df[[col for col in df.columns if col != 'merchant_category']].merge(merchant_df,on='merchant_id',how='left')

In [10]:
df.head()

,transaction_id,account_id,customer_id,timestamp,amount,merchant_id,channel,authentication_method,device_id,ip_id,transaction_state,fraud_label,fraud_type,fraud_discovery_date,existing_model_score,existing_model_prediction,account_age_days,account_type,credit_limit,age,annual_income,home_state,customer_tenure_days,device_type,device_age_days,ip_type,merchant_category
0,1828402,36178,95737,2024-01-08 07:01:49,61.74,1345,ECOMMERCE,contactless,55024.0,104754.0,TN,0,legitimate,NaN,0.225364,0,1182,debit_card,NaN,21,40496.87,TN,849,android,1425.0,residential,restaurant
1,1200072,52330,32499,2025-02-13 10:55:55,19.58,6159,ECOMMERCE,3ds,41220.0,83926.0,NY,0,legitimate,NaN,0.047296,0,1681,credit_card,6792.03,68,83693.19,NY,55,iphone,1400.0,residential,grocery
2,194850,16654,59925,2024-10-02 14:59:28,337.28,8986,MOBILE,password,6709.0,17431.0,LA,0,legitimate,NaN,0.043783,0,277,debit_card,NaN,27,57798.05,LA,946,mac,331.0,mobile,grocery
3,1629055,100442,74540,2024-09-09 06:32:08,43.37,9382,POS,contactless,4769.0,106315.0,MD,0,legitimate,NaN,0.014600,0,1278,credit_card,3271.88,18,138129.36,MD,533,unknown,1281.0,mobile,online_marketplace
4,191145,88323,83760,2024-05-26 05:09:17,17.83,6994,PHONE,contactless,23.0,1533.0,OK,0,legitimate,NaN,0.169942,0,850,credit_card,21004.56,27,150409.87,WV,81,android,375.0,corporate,restaurant


# Train-Validation-Test Split

In [13]:
df['timestamp'] = pd.to_datetime(df.timestamp)
df['monthyear'] = df.timestamp.dt.year.astype('str') + df.timestamp.dt.month.astype('str').str.zfill(2)

In [14]:
df['monthyear'].value_counts(1).sort_index().cumsum()

,proportion
monthyear,
202401,0.056725
202402,0.109769
202403,0.166499
202404,0.221543
202405,0.278479
202406,0.333485
202407,0.389961
202408,0.446789
202409,0.501686


In [15]:
df_train = df[(df.monthyear >= '202401') & (df.monthyear <= '202502')]
df_val = df[(df.monthyear >= '202503') & (df.monthyear <= '202504')]
df_test = df[(df.monthyear >= '202505') & (df.monthyear <= '202506')]
print(df_train.shape, df_val.shape, df_test.shape)

(1556312, 28) (223968, 28) (219720, 28)


# Exploratory Data Analysis

In [ ]:
df.head()

,transaction_id,account_id,customer_id,timestamp,amount,merchant_id,channel,authentication_method,device_id,ip_id,transaction_state,fraud_label,fraud_type,fraud_discovery_date,existing_model_score,existing_model_prediction,account_age_days,account_type,credit_limit,age,annual_income,home_state,customer_tenure_days,device_type,device_age_days,ip_type,merchant_category,monthyear
0,1828402,36178,95737,2024-01-08 07:01:49,61.74,1345,ECOMMERCE,contactless,55024.0,104754.0,TN,0,legitimate,NaN,0.225364,0,1182,debit_card,NaN,21,40496.87,TN,849,android,1425.0,residential,restaurant,202401
1,1200072,52330,32499,2025-02-13 10:55:55,19.58,6159,ECOMMERCE,3ds,41220.0,83926.0,NY,0,legitimate,NaN,0.047296,0,1681,credit_card,6792.03,68,83693.19,NY,55,iphone,1400.0,residential,grocery,202502
2,194850,16654,59925,2024-10-02 14:59:28,337.28,8986,MOBILE,password,6709.0,17431.0,LA,0,legitimate,NaN,0.043783,0,277,debit_card,NaN,27,57798.05,LA,946,mac,331.0,mobile,grocery,202410
3,1629055,100442,74540,2024-09-09 06:32:08,43.37,9382,POS,contactless,4769.0,106315.0,MD,0,legitimate,NaN,0.014600,0,1278,credit_card,3271.88,18,138129.36,MD,533,unknown,1281.0,mobile,online_marketplace,202409
4,191145,88323,83760,2024-05-26 05:09:17,17.83,6994,PHONE,contactless,23.0,1533.0,OK,0,legitimate,NaN,0.169942,0,850,credit_card,21004.56,27,150409.87,WV,81,android,375.0,corporate,restaurant,202405


In [ ]:
print(df_train.fraud_label.mean(),df_val.fraud_label.mean(),df_test.fraud_label.mean())

0.007123892895511954 0.003120981568795542 0.003763881303477153


In [ ]:
print(df_train.fraud_label.sum(),df_val.fraud_label.sum(),df_test.fraud_label.sum())

11087 699 827


# Feature Engineering

In [11]:
df.head()

,transaction_id,account_id,customer_id,timestamp,amount,merchant_id,channel,authentication_method,device_id,ip_id,transaction_state,fraud_label,fraud_type,fraud_discovery_date,existing_model_score,existing_model_prediction,account_age_days,account_type,credit_limit,age,annual_income,home_state,customer_tenure_days,device_type,device_age_days,ip_type,merchant_category
0,1828402,36178,95737,2024-01-08 07:01:49,61.74,1345,ECOMMERCE,contactless,55024.0,104754.0,TN,0,legitimate,NaN,0.225364,0,1182,debit_card,NaN,21,40496.87,TN,849,android,1425.0,residential,restaurant
1,1200072,52330,32499,2025-02-13 10:55:55,19.58,6159,ECOMMERCE,3ds,41220.0,83926.0,NY,0,legitimate,NaN,0.047296,0,1681,credit_card,6792.03,68,83693.19,NY,55,iphone,1400.0,residential,grocery
2,194850,16654,59925,2024-10-02 14:59:28,337.28,8986,MOBILE,password,6709.0,17431.0,LA,0,legitimate,NaN,0.043783,0,277,debit_card,NaN,27,57798.05,LA,946,mac,331.0,mobile,grocery
3,1629055,100442,74540,2024-09-09 06:32:08,43.37,9382,POS,contactless,4769.0,106315.0,MD,0,legitimate,NaN,0.014600,0,1278,credit_card,3271.88,18,138129.36,MD,533,unknown,1281.0,mobile,online_marketplace
4,191145,88323,83760,2024-05-26 05:09:17,17.83,6994,PHONE,contactless,23.0,1533.0,OK,0,legitimate,NaN,0.169942,0,850,credit_card,21004.56,27,150409.87,WV,81,android,375.0,corporate,restaurant


In [ ]:
def feature_engineering(df):
  # transaction velocity
  df = df.sort_values(by=['account_id','timestamp'])
  df['txn_cnt_5min']

# Feature Selection

# Base Model (Preexisting)

## Model Evaluation

In [16]:
df_train.existing_model_prediction

,transaction_id,account_id,customer_id,timestamp,amount,merchant_id,channel,authentication_method,device_id,ip_id,transaction_state,fraud_label,fraud_type,fraud_discovery_date,existing_model_score,existing_model_prediction,account_age_days,account_type,credit_limit,age,annual_income,home_state,customer_tenure_days,device_type,device_age_days,ip_type,merchant_category,monthyear
0,1828402,36178,95737,2024-01-08 07:01:49,61.74,1345,ECOMMERCE,contactless,55024.0,104754.0,TN,0,legitimate,NaN,0.225364,0,1182,debit_card,NaN,21,40496.87,TN,849,android,1425.0,residential,restaurant,202401
1,1200072,52330,32499,2025-02-13 10:55:55,19.58,6159,ECOMMERCE,3ds,41220.0,83926.0,NY,0,legitimate,NaN,0.047296,0,1681,credit_card,6792.03,68,83693.19,NY,55,iphone,1400.0,residential,grocery,202502
2,194850,16654,59925,2024-10-02 14:59:28,337.28,8986,MOBILE,password,6709.0,17431.0,LA,0,legitimate,NaN,0.043783,0,277,debit_card,NaN,27,57798.05,LA,946,mac,331.0,mobile,grocery,202410
3,1629055,100442,74540,2024-09-09 06:32:08,43.37,9382,POS,contactless,4769.0,106315.0,MD,0,legitimate,NaN,0.014600,0,1278,credit_card,3271.88,18,138129.36,MD,533,unknown,1281.0,mobile,online_marketplace,202409
4,191145,88323,83760,2024-05-26 05:09:17,17.83,6994,PHONE,contactless,23.0,1533.0,OK,0,legitimate,NaN,0.169942,0,850,credit_card,21004.56,27,150409.87,WV,81,android,375.0,corporate,restaurant,202405


# Logistic Regression

## Hyperparameter Tuning

## Train Model

## Model Evaluation

## Export Model

# XGBoost

## Hyperparameter Tuning

## Train Model

## Model Evaluation

## Export Model